In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy

In [4]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [5]:
X = df['clean_comment']
y = df['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [9]:
# Function to extract custom features
def extract_custom_features(text):
    doc = nlp(text)
    word_list = [token.text for token in doc if not token.is_punct]

    # 1. Comment Length
    comment_length = len(text)

    # 2. Word Count
    word_count = len(word_list)

    # 3. Average word length
    avg_word_length = sum(len(word) for word in word_list) / word_count if word_count > 0 else 0

    # 4. unique Word count
    unique_word_count = len(set(word_list))

    # 5. Lexical Diversity
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0 

    # 6. Count of POS Tags
    pos_count = len([token.pos_ for token in doc])

    # 7. Proportion of POS Tags
    pos_tags = [token.pos_ for token in doc]
    pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in set(pos_tags)} if word_count > 0 else {}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion # Flattening the POS proportions
    }


In [10]:
# Apply the custom feature extraction
train_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_train])
test_custom_features = pd.DataFrame([extract_custom_features(text) for text in X_test])

In [11]:
train_custom_features.head()

,comment_length,word_count,avg_word_length,unique_word_count,lexical_diversity,pos_count,VERB,NOUN,AUX,ADP,...,ADJ,PART,NUM,INTJ,CCONJ,SCONJ,X,DET,PUNCT,SYM
0,51,7,6.428571,7,1.000000,7,0.428571,0.142857,0.142857,0.142857,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,36,6,5.166667,6,1.000000,6,0.333333,0.333333,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,64,9,6.222222,9,1.000000,9,0.222222,0.444444,NaN,NaN,...,0.222222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,108,15,6.266667,14,0.933333,15,0.200000,0.466667,NaN,NaN,...,0.200000,0.066667,0.066667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,1,6.000000,1,1.000000,1,NaN,1.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Replace NaN values in POS tag proportion with 0
train_custom_features.fillna(0, inplace=True)
test_custom_features.fillna(0, inplace=True)

In [13]:
test_custom_features.isnull().sum()

comment_length       0
word_count           0
avg_word_length      0
unique_word_count    0
lexical_diversity    0
pos_count            0
VERB                 0
ADJ                  0
NOUN                 0
PRON                 0
ADP                  0
PROPN                0
AUX                  0
NUM                  0
ADV                  0
CCONJ                0
INTJ                 0
DET                  0
PART                 0
SCONJ                0
X                    0
PUNCT                0
SYM                  0
dtype: int64

In [14]:
# Apply TfidfVectorizer with trigam setting and max_features=1000
tfidf = TfidfVectorizer(ngram_range=(1, 3), max_features=10000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [15]:
# Convert TF-IDF to DataFrame
X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
X_test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), columns=tfidf.get_feature_names_out())

In [16]:
# Combine TF-IDF and custom features
X_train_combined = pd.concat([X_train_tfidf_df.reset_index(drop=True), train_custom_features.reset_index(drop=True)], axis=1)
X_test_combined = pd.concat([X_test_tfidf_df.reset_index(drop=True), test_custom_features.reset_index(drop=True)], axis=1)

In [17]:
X_train_combined

,000,000 000,000 crore,000 rupee,100,100 crore,100 time,100 year,1000,1000 note,...,ADJ,PART,NUM,INTJ,CCONJ,SCONJ,X,DET,PUNCT,SYM
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.222222,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.200000,0.066667,0.066667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29324,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.250000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29325,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.100000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29326,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
29327,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.189189,0.000000,0.000000,0.054054,0.0,0.0,0.0,0.0,0.0,0.0


In [18]:
import lightgbm as lgb

In [23]:
model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    metric="multi_logloss",
    is_unbalance= True,
    class_weight= "balanced",
    reg_alpha = 0.1, # L1 regularization
    reg_lambda = 0.1, # L2 regularization
    learning_rate= 0.08059218233839623,
    max_depth= 15,
    n_estimators= 496
)

In [24]:
# Fit the model on the resampled training data
model.fit(X_train_combined, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.116748 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 136999
[LightGBM] [Info] Number of data points in the train set: 29329, number of used features: 4461
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,15
,learning_rate,0.08059218233839623
,n_estimators,496
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [27]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_combined)
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.8130369562252829

In [29]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

          -1       0.75      0.74      0.74      1647
           0       0.76      0.98      0.85      2510
           1       0.92      0.72      0.81      3176

    accuracy                           0.81      7333
   macro avg       0.81      0.81      0.80      7333
weighted avg       0.83      0.81      0.81      7333

